## All I Want for Christmas Is You over Christmas 2025/26, north and south

Mariah's weekly position from mid-October 2025 to mid-January 2026, on Spotify's global chart and in the English-speaking countries of each hemisphere. The data comes from `pull_spotify_weekly.py`, which already covers these weeks. Each country has its own colour and the global chart is a thicker grey line. A line breaks wherever the song was out of that country's top 200. Drawn in the bwcharts light theme.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

# bwcharts isn't installed in this venv, so load it straight from its folder
BWCHARTS_DIR = r"C:\Users\Harry Turner\Documents\bwcharts"
if BWCHARTS_DIR not in sys.path:
    sys.path.insert(0, BWCHARTS_DIR)
import bwcharts as bw

bw.set_theme("light")

In [ ]:
# English-speaking countries on the charts.spotify.com dropdown, by hemisphere. Singapore, the
# Philippines, India, Nigeria and Pakistan have English as an official language but most people
# there speak another language day to day, so they're left out.
NORTH = ["United States", "United Kingdom", "Canada", "Ireland"]
SOUTH = ["Australia", "New Zealand", "South Africa"]
CHARTS = NORTH + SOUTH + ["Global"]

# the weeks shown: chart weeks ending from mid-October to mid-January
FIRST_WEEK, LAST_WEEK = pd.Timestamp("2025-10-16"), pd.Timestamp("2026-01-15")

weekly = pd.read_csv(Path("data") / "spotify_weekly_all_regions.csv", parse_dates=["week_end"],
                     dtype={"streams": str}, keep_default_na=False, na_values={"rank": [""]})
weekly = weekly[weekly["region_name"].isin(CHARTS)
                & weekly["week_end"].between(FIRST_WEEK, LAST_WEEK)]

# Mariah's original only, matched on title and artist (this leaves out the Justin Bieber duet)
is_mariah = ((weekly["track_name"] == "All I Want for Christmas Is You")
             & (weekly["artists"] == "Mariah Carey"))

# one row per chart week, one column per chart, empty when the song wasn't in the top 200
all_weeks = pd.DatetimeIndex(sorted(weekly["week_end"].unique()))
rank = (weekly[is_mariah].pivot(index="week_end", columns="region_name", values="rank")
        .reindex(index=all_weeks, columns=CHARTS))

# a chart week runs Friday to Thursday, so each week is drawn at its middle (the Monday)
mid = all_weeks - pd.Timedelta(days=3)

# check the story: weeks in the top 200, best position, first and last week in it
pd.DataFrame({
    "hemisphere": ["North" if c in NORTH else "South" if c in SOUTH else "" for c in CHARTS],
    "weeks_in_top_200": rank.notna().sum(),
    "best": rank.min(),
    "first_week": rank.apply(lambda s: s.first_valid_index()).dt.strftime("%d %b"),
    "last_week": rank.apply(lambda s: s.last_valid_index()).dt.strftime("%d %b"),
}, index=CHARTS)

In [ ]:
# one colour per country from the bwcharts light palette, northern countries first; global is grey
COLOURS = {
    "United States": bw.C1, "United Kingdom": bw.C2, "Canada": bw.C3, "Ireland": bw.C4,
    "Australia": bw.C5, "New Zealand": bw.C6, "South Africa": bw.C7, "Global": bw.C8,
}
START = pd.Timestamp("2025-10-01")              # x positions are days since 1 October
x = (mid - START).days

fig, ax = bw.chart(
    "Mariah's Christmas, north and south, *working title*",        # rewrite once we've agreed the story
    "Weekly position of All I Want for Christmas Is You on Spotify's top 200, "
    "mid-October 2025 to mid-January 2026",
    note="Note: Weeks run Friday to Thursday and are drawn at their middle. A line breaks "
         "when the song is out of the top 200.",
    source="Source: Spotify Charts, weekly top 200",
    size="landscape", grid="y",
)

for name in CHARTS:
    is_global = name == "Global"
    ax.plot(x, rank[name], color=COLOURS[name], linewidth=2.6 if is_global else 1.4,
            zorder=1 if is_global else 2, solid_capstyle="round", solid_joinstyle="round")
    # a single week between two gaps draws no line, so mark every week with a dot too
    ax.scatter(x, rank[name], s=8, color=COLOURS[name], zorder=3, clip_on=False)

# Christmas Day
xmas = (pd.Timestamp("2025-12-25") - START).days
ax.axvline(xmas, color=bw.TEXT, linewidth=0.6, linestyle=(0, (2, 2)), zorder=0)
ax.annotate("Christmas Day", xy=(xmas, 1), xytext=(0, 6), textcoords="offset points",
            ha="center", va="bottom", fontsize=6.5, fontweight="semibold", color=bw.TEXT,
            annotation_clip=False)

# axes: No. 1 at the top, the first of each month along the bottom
ax.set_ylim(200, 1)
ax.set_yticks([1, 50, 100, 150, 200], ["No. 1", "50", "100", "150", "200"])
months = pd.to_datetime(["2025-11-01", "2025-12-01", "2026-01-01"])
ax.set_xticks((months - START).days, [f"{m.day} {m:%b}" for m in months])
ax.set_xlim((FIRST_WEEK - pd.Timedelta(days=7) - START).days,
            (LAST_WEEK + pd.Timedelta(days=1) - START).days)

# a key above the chart, north first, then south, then global
bw.key(ax, COLOURS)

Path("charts").mkdir(exist_ok=True)
bw.save(fig, "charts/aiwfc_weekly_christmas_2025_26_north_south_light.png")

In [ ]:
# Small multiples: one panel per country, northern hemisphere on the top row and southern
# below, with the global chart as a grey line behind each country for comparison
panels = NORTH + SOUTH                          # four on top, three below; the last cell is empty
fig, axes = bw.facets(
    len(panels), ncols=4,
    title="Mariah's Christmas, north and south, *working title*",  # rewrite once we've agreed the story
    subtitle="Weekly position of All I Want for Christmas Is You on Spotify's top 200, 2025/26. "
             "Grey line: the global chart",
    note="Note: Weeks run Friday to Thursday. A line breaks when the song is out of the top 200. "
         "Dotted line: Christmas Day.",
    source="Source: Spotify Charts, weekly top 200",
    panel_titles=panels,
    panel_subtitles=["Northern hemisphere" if p in NORTH else "Southern hemisphere" for p in panels],
    sharex=True, sharey=True, size="landscape", wspace=0.15,
)

for ax, name in zip(axes, panels):
    ax.plot(x, rank["Global"], color=COLOURS["Global"], linewidth=1.8, zorder=1,
            solid_capstyle="round", solid_joinstyle="round")
    ax.plot(x, rank[name], color=COLOURS[name], linewidth=1.6, zorder=2,
            solid_capstyle="round", solid_joinstyle="round")
    # a single week between two gaps draws no line, so mark every week with a dot too
    ax.scatter(x, rank[name], s=7, color=COLOURS[name], zorder=3, clip_on=False)
    ax.axvline(xmas, color=bw.TEXT, linewidth=0.5, linestyle=(0, (2, 2)), zorder=0)

# shared axes, so setting them once sets every panel
axes[0].set_ylim(200, 1)
axes[0].set_yticks([1, 100, 200], ["No. 1", "100", "200"])
axes[0].set_xticks((months - START).days, [f"{m:%b}" for m in months])
axes[0].set_xlim((FIRST_WEEK - pd.Timedelta(days=7) - START).days,
                 (LAST_WEEK + pd.Timedelta(days=1) - START).days)

bw.save(fig, "charts/aiwfc_weekly_christmas_2025_26_small_multiples_light.png")